In [1]:
import os, sys, glob, warnings
warnings.filterwarnings("ignore")           # hides the PraatWarning flood that filled the IEMOCAP log
!pip install -q praat-parselmouth seaborn
!git clone -q https://github.com/zehuiwu/SpeechCueLLM /kaggle/working/SpeechCueLLM
os.chdir("/kaggle/working/SpeechCueLLM"); sys.path.insert(0, "feature_utils")
DEN = "/kaggle/input/notebooks/kushalag21/denoising-datasets/denoised/meld"
print(len(glob.glob(DEN + "/*.wav")), "denoised MELD wavs")        # expect 13848
print(sorted(os.listdir(DEN))[:3])

13833 denoised MELD wavs
['MELD.Raw__dev__dev_splits_complete__dia0_utt0.wav', 'MELD.Raw__dev__dev_splits_complete__dia0_utt1.wav', 'MELD.Raw__dev__dev_splits_complete__dia100_utt0.wav']


In [2]:
import pandas as pd, numpy as np
from multiprocessing import Pool
from tqdm import tqdm
from extract_audio_feature import extract_audio_features_with_praat

FOLDER = {"train": "train_splits", "dev": "dev_splits_complete", "test": "output_repeated_splits_test"}
COLS = ["filename","duration","avg_intensity","intensity_variation","avg_pitch","pitch_std","pitch_range","articulation_rate","mean_hnr"]
os.makedirs("/kaggle/working/B2_meld", exist_ok=True)

def work(p):
    warnings.filterwarnings("ignore")
    try:
        d = extract_audio_features_with_praat(p); d.pop("filename"); return d
    except Exception:
        return {}

for split in ["train", "dev", "test"]:
    ref = pd.read_csv(f"speech_features/meld_processed_5class_{split}_audio_features.csv")
    names = ref["filename"].tolist()                      # e.g. dia0_utt0.wav
    paths = [f"{DEN}/MELD.Raw__{split}__{FOLDER[split]}__{n}" for n in names]
    missing = sum(not os.path.exists(p) for p in paths)
    print(split, len(paths), "missing denoised files:", missing)
    with Pool(4) as pool:
        feats = list(tqdm(pool.imap(work, paths, chunksize=20), total=len(paths), mininterval=30))
    df = pd.concat([pd.Series(names, name="filename"), pd.DataFrame(feats)], axis=1)[COLS]
    df.to_csv(f"/kaggle/working/B2_meld/meld_{split}_audio_features.csv", index=False)
    print(split, df.shape, "| empty rows:", df["avg_intensity"].isna().sum())

train 9988 missing denoised files: 12


100%|██████████| 9988/9988 [11:30<00:00, 14.47it/s]


train (9988, 9) | empty rows: 12
dev 1112 missing denoised files: 2


100%|██████████| 1112/1112 [01:17<00:00, 14.31it/s]


dev (1112, 9) | empty rows: 2
test 2615 missing denoised files: 0


100%|██████████| 2615/2615 [03:16<00:00, 13.30it/s]

test (2615, 9) | empty rows: 0


In [3]:
import glob
miss = []
for split, folder in FOLDER.items():
    ref = pd.read_csv(f"speech_features/meld_processed_5class_{split}_audio_features.csv")
    for n, dur in zip(ref["filename"], ref["duration"]):
        if not os.path.exists(f"{DEN}/MELD.Raw__{split}__{folder}__{n}"):
            miss.append((split, n, round(dur, 3)))
print(len(miss)); print(miss)
!head -3 /kaggle/input/notebooks/kushalag21/denoising-datasets/denoised/meld_fails.txt

14
[('train', 'dia4_utt5.wav', 0.1), ('train', 'dia71_utt0.wav', 0.08), ('train', 'dia179_utt8.wav', 0.08), ('train', 'dia312_utt9.wav', 0.08), ('train', 'dia332_utt1.wav', 0.08), ('train', 'dia345_utt0.wav', 0.1), ('train', 'dia392_utt2.wav', 0.1), ('train', 'dia406_utt2.wav', 0.08), ('train', 'dia523_utt10.wav', 0.08), ('train', 'dia598_utt3.wav', 0.11), ('train', 'dia739_utt7.wav', 0.1), ('train', 'dia747_utt8.wav', 0.1), ('dev', 'dia99_utt3.wav', 0.08), ('dev', 'dia108_utt5.wav', 0.08)]


In [4]:
import postprocess_audio_feature_meld as pp
NC = 5

# the repo's functions read speaker info from a *_sent_emo.csv; rebuild one from the raw processed files
for s in ["train", "dev", "test"]:
    r = pd.read_csv(f"speech_features/meld_processed_5class_{s}_audio_features.csv")
    pd.DataFrame({"Dialogue_ID": r.Dialogue_ID, "Utterance_ID": r.Utterance_ID,
                  "Speaker": r.speaker, "Utterance": r.Utterance}).to_csv(f"/kaggle/working/B2_meld/spk_{s}.csv", index=False)

# raw thresholds, only for the comparison table
thr_raw, _ = pp.extract_thresholds_and_stats("speech_features/meld_train_audio_features.csv", "/kaggle/working/B2_meld/spk_train.csv", NC)
!mkdir -p /kaggle/working/B2_meld/raw_plots && mv feature_distributions_*.png /kaggle/working/B2_meld/raw_plots/

# denoised thresholds from the denoised training split
thr, st = pp.extract_thresholds_and_stats("/kaggle/working/B2_meld/meld_train_audio_features.csv", "/kaggle/working/B2_meld/spk_train.csv", NC)
drop = ["avg_intensity","intensity_variation","avg_pitch","pitch_std","pitch_range","articulation_rate","mean_hnr"]
for s in ["train", "dev", "test"]:
    df = pp.standardize_and_process_csv(f"/kaggle/working/B2_meld/meld_{s}_audio_features.csv", thr, st, f"/kaggle/working/B2_meld/spk_{s}.csv", NC)
    df["description"] = df.apply(lambda r: pp.generate_concise_description(r, NC), axis=1)
    df["impression"]  = df.apply(lambda r: pp.generate_impression(r, NC), axis=1)
    out = df[[c for c in df.columns if c not in drop]]
    out.to_csv(f"/kaggle/working/B2_meld/meld_processed_5class_{s}_audio_features.csv", index=False)

    raw = pd.read_csv(f"speech_features/meld_processed_5class_{s}_audio_features.csv")
    m = out.merge(raw, on="filename", suffixes=("_den", "_raw"))
    print(s, out.shape, raw.shape, "| same columns:", list(out.columns) == list(raw.columns))
    for c in ["avg_intensity_category","intensity_variation_category","pitch_std_category","articulation_rate_category"]:
        print("   ", c, "same bucket as raw: %.1f%%" % (100*(m[c+"_den"] == m[c+"_raw"]).mean()))
print(out["description"].iloc[0]); print(raw["description"].iloc[0])

rows = [(f, k, round(thr_raw["overall"][f][k], 2), round(thr["overall"][f][k], 2))
        for f in ["avg_intensity","intensity_variation","avg_pitch","pitch_std","articulation_rate"]
        for k in ["very_low","low","medium","high"]]
print(pd.DataFrame(rows, columns=["feature","cutoff","raw","denoised"]).to_string(index=False))
!cp feature_distributions_*.png /kaggle/working/B2_meld/

train (9988, 24) (9988, 24) | same columns: True
    avg_intensity_category same bucket as raw: 94.9%
    intensity_variation_category same bucket as raw: 42.9%
    pitch_std_category same bucket as raw: 50.3%
    articulation_rate_category same bucket as raw: 42.8%
dev (1112, 24) (1112, 24) | same columns: True
    avg_intensity_category same bucket as raw: 88.2%
    intensity_variation_category same bucket as raw: 41.6%
    pitch_std_category same bucket as raw: 49.0%
    articulation_rate_category same bucket as raw: 42.8%
test (2615, 24) (2615, 24) | same columns: True
    avg_intensity_category same bucket as raw: 88.0%
    intensity_variation_category same bucket as raw: 44.2%
    pitch_std_category same bucket as raw: 48.3%
    articulation_rate_category same bucket as raw: 42.0%
Target speech characteristics: low volume with moderate variation, low pitch with moderate variation, very high speaking rate.
Target speech characteristics: low volume with moderate variation, low pitc

In [5]:
import pandas as pd
for s in ["train", "dev", "test"]:
    d = pd.read_csv(f"/kaggle/working/B2_meld/meld_processed_5class_{s}_audio_features.csv")
    print(s, d.shape, "insufficient-data rows:", d["description"].str.startswith("Insufficient").sum())

train (9988, 24) insufficient-data rows: 12
dev (1112, 24) insufficient-data rows: 2
test (2615, 24) insufficient-data rows: 0


In [6]:
!cd /kaggle/working/B2_meld && mkdir -p plots && mv feature_distributions_*.png plots/ 2>/dev/null; rm -f spk_*.csv
!ls -R /kaggle/working/B2_meld | head -40

/kaggle/working/B2_meld:
meld_dev_audio_features.csv
meld_processed_5class_dev_audio_features.csv
meld_processed_5class_test_audio_features.csv
meld_processed_5class_train_audio_features.csv
meld_test_audio_features.csv
meld_train_audio_features.csv
plots
raw_plots

/kaggle/working/B2_meld/plots:
feature_distributions_overall.png
feature_distributions_speaker_Chandler.png
feature_distributions_speaker_Joey.png
feature_distributions_speaker_Monica.png
feature_distributions_speaker_Phoebe.png
feature_distributions_speaker_Rachel.png
feature_distributions_speaker_Ross.png

/kaggle/working/B2_meld/raw_plots:
feature_distributions_overall.png
feature_distributions_speaker_Chandler.png
feature_distributions_speaker_Joey.png
feature_distributions_speaker_Monica.png
feature_distributions_speaker_Phoebe.png
feature_distributions_speaker_Rachel.png
feature_distributions_speaker_Ross.png
